# Course 4: Large Language Models
## Session 10: Beyond autoregressive transformers

**Goal:** train a masked diffusion language model and an autoregressive model of similar size on the same data, and compare what they generate.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

### 1. The autoregressive baseline

In [ ]:
import math, numpy as np
cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=64)
ar = train_lm(cfg, steps=600)
ar_samples = sample_sentences(ar, 500)
n_ar = sum(p.numel() for p in ar.parameters())
print("AR model: %d parameters | valid %.3f | distinct %d of 500" % (n_ar, sum(map(is_valid, ar_samples)) / 500, len({tuple(s) for s in ar_samples})))

### 2. A masked diffusion language model
Training: pick a noise level t between 0 and 1, replace each token with `<mask>` with probability t, and train a bidirectional transformer to fill in the masked tokens. The loss on masked tokens is weighted by 1/t. This is the recipe used by MDLM and LLaDA.

Generation: start from a sentence that is all masks and reveal tokens over several steps.

In [ ]:
MASK = STOI["<mask>"]
class MaskedLM(nn.Module):
    def __init__(self, vocab, d=64, layers=2, heads=2, length=6):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(length, d)
        self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 4 * d, dropout=0.0, batch_first=True, norm_first=True), layers)
        self.head = nn.Linear(d, vocab)
    def forward(self, x): return self.head(self.enc(self.tok(x) + self.pos(torch.arange(x.size(1)))))       # no causal mask: sees both sides

torch.manual_seed(0)
mdm = MaskedLM(len(VOCAB)); opt = torch.optim.AdamW(mdm.parameters(), lr=3e-3)
S = torch.tensor([enc(s) for s in train_sents])
for step in range(1500):
    x0 = S[torch.randint(len(S), (64,))]; t = torch.rand(64, 1).clamp(min=0.05)
    masked = torch.rand(64, 6) < t; masked[torch.arange(64), torch.randint(6, (64,))] = True               # at least one mask per row
    xt = torch.where(masked, MASK, x0)
    ce = F.cross_entropy(mdm(xt).transpose(1, 2), x0, reduction="none")
    loss = ((ce * masked) / t).sum() / masked.sum(); opt.zero_grad(); loss.backward(); opt.step()
print("masked diffusion model: %d parameters (AR model: %d)" % (sum(p.numel() for p in mdm.parameters()), n_ar))

### 3. Sampling, and why the number of steps matters
With one step, all six tokens are sampled at the same time, each from its own distribution. The object and the verb are then chosen independently and often do not match. With more steps, later tokens can see earlier choices.

In [ ]:
@torch.no_grad()
def mdm_sample(n, steps, order="random"):
    x = torch.full((n, 6), MASK)
    per_step = [len(c) for c in np.array_split(np.arange(6), steps)]
    for k in per_step:
        probs = torch.softmax(mdm(x), -1); probs[:, :, MASK] = 0
        draw = torch.multinomial(probs.view(-1, probs.size(-1)), 1).view(n, 6)
        still = x == MASK
        if order == "random": score = torch.rand(n, 6)
        else: score = probs.gather(-1, draw[..., None]).squeeze(-1)             # "confident": reveal the easiest positions first
        score = score.masked_fill(~still, -1.0)
        pick = score.topk(k, dim=1).indices
        x.scatter_(1, pick, draw.gather(1, pick))
    return [dec(r.tolist()) for r in x]

torch.manual_seed(0)
rows = {}
for steps in [1, 2, 3, 6]:
    s = mdm_sample(500, steps); rows[steps] = sum(map(is_valid, s)) / 500
    print(f"masked diffusion, {steps} step(s): valid {rows[steps]:.3f} | distinct {len({tuple(w) for w in s})} of 500")
print("autoregressive (6 steps)   : valid %.3f" % (sum(map(is_valid, ar_samples)) / 500))
assert rows[6] > rows[1] + 0.3

This is the central trade-off. A diffusion language model can produce many tokens per step, which is where its speed comes from, but tokens produced in the same step do not see each other.

### 4. Any-order generation: which positions does the model fill first?
Let the model reveal its most confident positions first and record the order.

In [ ]:
@torch.no_grad()
def reveal_order(n=300):
    x = torch.full((n, 6), MASK); order = torch.zeros(n, 6, dtype=torch.long)
    for step in range(6):
        probs = torch.softmax(mdm(x), -1); probs[:, :, MASK] = 0
        conf, draw = probs.max(-1); conf = conf.masked_fill(x != MASK, -1.0)
        pick = conf.argmax(1); x[torch.arange(n), pick] = draw[torch.arange(n), pick]; order[torch.arange(n), pick] = step
    return order.float().mean(0)
o = reveal_order()
for name, v in zip(["subject", "time", "place", "object", "verb", "full stop"], o.tolist()): print(f"{name:10s} revealed at step {v:.1f} on average")
assert o[5] < 1.0

The model fills the certain position first (the full stop) and leaves the open choices for later. 'The Flexibility Trap' (ICML 2026 Outstanding Paper) argues that on reasoning tasks this freedom lets diffusion models put off the hard tokens and lose diversity in their solutions. The table above is a small picture of the behaviour that argument starts from.

### 5. Infilling: something the autoregressive model cannot do directly
Fix the first and last words and let the diffusion model fill the middle.

In [ ]:
torch.manual_seed(0)
x = torch.full((5, 6), MASK); x[:, 0] = STOI["দাদু"]; x[:, 4] = STOI["পড়ে"]; x[:, 5] = STOI["।"]
with torch.no_grad():
    for _ in range(3):
        probs = torch.softmax(mdm(x), -1); probs[:, :, MASK] = 0
        still = x == MASK; conf, draw = probs.max(-1)
        sampled = torch.multinomial(probs.view(-1, probs.size(-1)), 1).view(5, 6)
        pick = conf.masked_fill(~still, -1.0).argmax(1); x[torch.arange(5), pick] = sampled[torch.arange(5), pick]
filled = [dec(r.tolist()) for r in x]
for s in filled: print("  ", " ".join(s))
print("every filled sentence uses the object that fits the fixed verb:", all(s[3] == "বই" for s in filled))

### 6. The wider map
- State-space models such as Mamba replace attention with a learned recurrence whose cost grows linearly with length. They are attractive for very long sequences (audio, DNA), while in-context retrieval has been their weak point.
- Hybrid models mix a few attention layers into a state-space stack.
- Diffusion language models (LLaDA and others) are now competitive with autoregressive models of similar size on several benchmarks, with open questions on reasoning and on variable-length output.

None of these has displaced the autoregressive transformer. The useful habit is to ask of each: what does it buy (speed, length, editing), and what does it give up?

In [ ]:
# NOT RUN: needs a GPU and model downloads. Compare a small open masked diffusion LM with an autoregressive model of similar size.
# 1. Choose 30 prompts: 10 open-ended, 10 infilling (text before and after a gap), 10 short reasoning questions.
# 2. Generate with each model. For the diffusion model use 8, 32 and 128 steps.
# 3. Score: fluency by hand on a 1-5 scale (blind), exact match on the reasoning questions, and seconds per answer.

### TODO
1. Train the masked model without the 1/t weight. Does sample quality change?
2. In section 3, compare `order="random"` with `order="confident"` at 2 and 3 steps.
3. A linear recurrence `h_t = a * h_{t-1} + b * x_t` is the heart of a state-space layer. Implement it, then make `a` depend on `x_t` (the 'selective' part of Mamba) and test both on copying a token from 50 steps back.